In [19]:
from pyspark.sql import SparkSession

try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("engine")
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "/home/jovyan/notebooks/lakehouse")
    .getOrCreate()
)

spark.sql("CREATE NAMESPACE IF NOT EXISTS local.db_lakehouse")



DataFrame[]

# 1.- Extracción

In [20]:
jdbc_url = "jdbc:postgresql://postgres:5432/db_acatlan"
props = {
    "user": "acatlan",
    "password": "8g4izJPX",
    "driver": "org.postgresql.Driver",
}

df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "public.order_items")
    .option("user", props["user"])
    .option("password", props["password"])
    .option("driver", props["driver"])
    .load()
)



# 2.- Transformacion

In [21]:
df.show()

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35|  58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13| 239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30| 199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18| 12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51| 199.9|        18.14|
|00048cc3ae777c65d...|            1|ef92

# 2.- Carga

In [22]:
df.writeTo("local.db_lakehouse.order_items").createOrReplace()

# 3 .- ACID

In [23]:
spark.sql("SELECT * FROM local.db_lakehouse.order_items LIMIT 4").show()

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+



### Insert

In [24]:
spark.sql("""
INSERT INTO local.db_lakehouse.order_items
VALUES (
    'pedido-demo',
    1,
    'producto-demo',
    'vendedor-demo',
    TIMESTAMP '2026-11-17 12:00:00',
    99.9,
    12.5
)
""")

DataFrame[]

In [25]:
spark.sql("""
SELECT * FROM local.db_lakehouse.order_items
WHERE order_id = 'pedido-demo'
""").show()

+-----------+-------------+-------------+-------------+-------------------+-----+-------------+
|   order_id|order_item_id|   product_id|    seller_id|shipping_limit_date|price|freight_value|
+-----------+-------------+-------------+-------------+-------------------+-----+-------------+
|pedido-demo|            1|producto-demo|vendedor-demo|2026-11-17 12:00:00| 99.9|         12.5|
+-----------+-------------+-------------+-------------+-------------------+-----+-------------+



### Update

In [26]:
spark.sql("""
UPDATE local.db_lakehouse.order_items
SET price = 150.0,
    freight_value = 20.0
WHERE order_id = 'pedido-demo'
""")

DataFrame[]

In [27]:
spark.sql("""
SELECT order_id, price, freight_value
FROM local.db_lakehouse.order_items
WHERE order_id = 'pedido-demo'
""").show()

+-----------+-----+-------------+
|   order_id|price|freight_value|
+-----------+-----+-------------+
|pedido-demo|150.0|         20.0|
+-----------+-----+-------------+



### Delete

In [28]:
spark.sql("""
DELETE FROM local.db_lakehouse.order_items
WHERE order_id = 'pedido-demo'
""")

DataFrame[]

In [29]:
spark.sql("""
SELECT * FROM local.db_lakehouse.order_items
WHERE order_id = 'pedido-demo'
""").show()

+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
+--------+-------------+----------+---------+-------------------+-----+-------------+



### Truncate 

In [30]:
spark.sql("TRUNCATE TABLE local.db_lakehouse.order_items")

DataFrame[]

In [31]:
spark.sql("SELECT COUNT(*) FROM local.db_lakehouse.order_items").show()

+--------+
|count(1)|
+--------+
|       0|
+--------+



### Drop

In [32]:
spark.sql("DROP TABLE local.db_lakehouse.order_items")

DataFrame[]

In [33]:
spark.sql("SHOW TABLES IN local.db_lakehouse").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+

